In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (f1_score, matthews_corrcoef,
                             average_precision_score, roc_auc_score,
                             confusion_matrix, classification_report,
                             precision_recall_curve)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

test = pd.read_csv('../data/test_final.csv')
X_test = test.drop(columns=['isFraud'])
y_test = test['isFraud']

best_model = xgb.XGBClassifier()
best_model.load_model('../models/xgb_tuned.json')

with open('../models/best_threshold.json') as f:
    threshold = json.load(f)['threshold']

y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

print(f"Test size: {X_test.shape}")
print(f"Threshold: {threshold}")
print(f"Fraud trong test: {y_test.sum()}")

Test size: (2000, 192)
Threshold: 0.1
Fraud trong test: 77


In [2]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negative  (Legit đúng):   {tn}")
print(f"False Positive (Legit → Fraud): {fp}  ← False alarm")
print(f"False Negative (Fraud → Legit): {fn}  ← Bỏ sót fraud!")
print(f"True Positive  (Fraud đúng):    {tp}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.title(f'Confusion Matrix (threshold={threshold})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../reports/confusion_matrix.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu confusion matrix!")

True Negative  (Legit đúng):   1913
False Positive (Legit → Fraud): 10  ← False alarm
False Negative (Fraud → Legit): 48  ← Bỏ sót fraud!
True Positive  (Fraud đúng):    29
✅ Đã lưu confusion matrix!


In [3]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PR Curve
precision_arr, recall_arr, _ = precision_recall_curve(y_test, y_prob)
auprc = average_precision_score(y_test, y_prob)

axes[0].plot(recall_arr, precision_arr, color='purple', lw=2)
axes[0].fill_between(recall_arr, precision_arr, alpha=0.2, color='purple')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall Curve\nAUPRC = {auprc:.4f}')
axes[0].grid(True, alpha=0.3)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

axes[1].plot(fpr, tpr, color='coral', lw=2)
axes[1].fill_between(fpr, tpr, alpha=0.2, color='coral')
axes[1].plot([0,1], [0,1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve\nAUC = {auc:.4f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/pr_roc_curves.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu PR + ROC curves!")

✅ Đã lưu PR + ROC curves!


In [5]:
# Tạo bảng kết quả thủ công từ các bước đã chạy
all_results = pd.DataFrame([
    {'model': 'Logistic Regression', 'F1': 0.1074, 'MCC': 0.0851, 'AUPRC': 0.0830, 'AUC': 0.6414},
    {'model': 'Decision Tree',       'F1': 0.4421, 'MCC': 0.4238, 'AUPRC': 0.2236, 'AUC': 0.7415},
    {'model': 'LSTM',                'F1': 0.2388, 'MCC': 0.2156, 'AUPRC': 0.2229, 'AUC': 0.6694},
    {'model': 'XGBoost Default',     'F1': 0.3579, 'MCC': 0.4487, 'AUPRC': 0.5649, 'AUC': 0.8812},
    {'model': 'LightGBM',            'F1': 0.3958, 'MCC': 0.4894, 'AUPRC': 0.5574, 'AUC': 0.8543},
    {'model': 'XGBoost Tuned',       'F1': 0.4554, 'MCC': 0.5269, 'AUPRC': 0.5813, 'AUC': 0.8918},
    {
        'model': 'XGBoost Tuned + Threshold',
        'F1':    round(f1_score(y_test, y_pred), 4),
        'MCC':   round(matthews_corrcoef(y_test, y_pred), 4),
        'AUPRC': round(average_precision_score(y_test, y_prob), 4),
        'AUC':   round(roc_auc_score(y_test, y_prob), 4)
    }
])

print("📊 BẢNG TỔNG HỢP KẾT QUẢ:")
print("="*65)
print(all_results.to_string(index=False))

best_idx = all_results['F1'].idxmax()
print(f"\n🏆 Model tốt nhất: {all_results.loc[best_idx, 'model']}")
print(f"   F1:    {all_results.loc[best_idx, 'F1']:.4f}")
print(f"   AUPRC: {all_results.loc[best_idx, 'AUPRC']:.4f}")

all_results.to_csv('../reports/final_results.csv', index=False)
print("✅ Đã lưu final_results.csv!")

📊 BẢNG TỔNG HỢP KẾT QUẢ:
                    model     F1    MCC  AUPRC    AUC
      Logistic Regression 0.1074 0.0851 0.0830 0.6414
            Decision Tree 0.4421 0.4238 0.2236 0.7415
                     LSTM 0.2388 0.2156 0.2229 0.6694
          XGBoost Default 0.3579 0.4487 0.5649 0.8812
                 LightGBM 0.3958 0.4894 0.5574 0.8543
            XGBoost Tuned 0.4554 0.5269 0.5813 0.8918
XGBoost Tuned + Threshold 0.5000 0.5168 0.5813 0.8918

🏆 Model tốt nhất: XGBoost Tuned + Threshold
   F1:    0.5000
   AUPRC: 0.5813
✅ Đã lưu final_results.csv!


In [6]:
metrics = ['F1', 'AUPRC', 'AUC', 'MCC']
models  = all_results['model'].tolist()
x = np.arange(len(metrics))
width = 0.12

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['steelblue','coral','green','purple','orange','red','darkblue']

for i, (_, row) in enumerate(all_results.iterrows()):
    ax.bar(x + i*width, row[metrics], width,
           label=row['model'], color=colors[i % len(colors)])

ax.set_xticks(x + width * len(models)/2)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Score')
ax.set_title('So sánh tất cả models — Final Report')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../reports/final_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu final comparison chart!")

✅ Đã lưu final comparison chart!
